## Introduction

*CartPole* is a classic reinforcement learning control task.

Imagine a cart that can move left or right on a track with a pole attached to the cart. The pole starts upright, and the goal is to keep it from falling

The agent can usually take two actions:

1.   move the cart left
2.   move the cart right

At each step, the environment gives a state with information of cart position
, cart velocity, pole angle, and pole angular velocity. The goal of this task is to keep the pole balanced upright for as long as possible, so a better policy keeps the pole balanced longer and gets more total reward.

## Objective

The scope of this project is to train a deep neural network to look at the cart-and-pole state and decide whether to move left or right so the pole stays balanced as long as possible. This approach is also known as **Policy gradient method implemented as iterative classification**

   

## Policy gradient method implemented as iterative classification

Policy gradient method essentially trains a classification model with the above loss function, but importantly - on data collected iteratively collected using the updated policy.

Given classification task with features $x_i$, labels $(y_i)$ ($1$ or $0$) and predictions $p_i$ (probability of classifiying as 1), recall  loss function for binary classification (Log loss)
$$-\frac{1}{N} \sum_i y_i \log(p_i) + (1-y_i) \log(1-p_i)$$
This can be extended to multi-label classification with $A$ classes; $y_{ij}$ is the label for class $j$ and $p_{ij}$ is the prediction (probability of classifiying the $i$th point as class $j$
$$-\frac{1}{N} \sum_i \sum_j y_{ij} \log(p_{ij})$$

We extend this to reinforcement learning, where given a data point with state features $(s_i)$, we predict the action probaility $\pi_\theta(s_i,a)$ for all $a\in A$. Howeer we do not know tHe true "label" , i.e. we do not know which action is optimal. We use that optimal action maximizes $Q^*(s_i,a)$ approximate the label $y_{ia}$ as $\hat{Q}(s_i,a)$: the $Q$ value under the current policy.

Thus, given samples of state action pairs $(s_i, a_i)$ and estimate $Q_i$ for $Q^{\pi_\phi}(s_i, a_i)$, for $i=1,\ldots, $ we  set the loss function  as
$$-\frac{1}{N} \sum_{i=1}^N Q_i \log(\pi_\theta(s_i, a_i)) $$
or more commonly we use the advantage function estimate $\hat{A}(s_i,a)=\hat{Q}(s_i,a)-\hat{V}(s_i,a)$ which essentially subtracts a baseline from $Q$ function estimates.


In [1]:
import torch
import numpy as np
import gymnasium as gym

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt



## Model implemntation (DNN)

In [2]:
class Policy(object):

    def __init__(self, obssize, actsize, lr):
        """
        obssize: size of the states
        actsize: size of the actions
        """
        # DEFINE THE MODEL
        self.model = torch.nn.Sequential(
                    torch.nn.Linear(obssize, 128),
                    torch.nn.ReLU(),
                    torch.nn.Linear(128, 128),
                    torch.nn.ReLU(),
                    torch.nn.Linear(128, actsize)
                )

        # DEFINE THE OPTIMIZER
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

        # RECORD HYPER-PARAMS
        self.obssize = obssize
        self.actsize = actsize

        # TEST
        self.compute_prob(np.random.randn(obssize).reshape(1, -1))



    def compute_prob(self, state):
        """
        compute prob over all actions given state: i.e., pi(s,a) for all a,
        as softmax of scores given by the neural network.
        """
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        prob = torch.nn.functional.softmax(self.model(state_tensor), dim=-1)
        return prob

    def compute_log_prob_selected(self, state, action):
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        logits = self.model(state_tensor)
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        action_tensor = torch.LongTensor([[action]])
        log_prob_selected = torch.gather(log_probs, 1, action_tensor).squeeze()
        return log_prob_selected

In [3]:
def discounted_rewards(r, gamma):
    """ take 1D float array of rewards and compute discounted reward """
    discounted_r = np.zeros_like(r)
    running_sum = 0
    for i in reversed(range(0,len(r))):
        discounted_r[i] = running_sum * gamma + r[i]
        running_sum = discounted_r[i]
    return list(discounted_r)

## Train the policy model using classification loss function

In [4]:
# parameter initializations
alpha = 5e-4
episodes = 700
envname = "CartPole-v0"
gamma = 0.99

# initialize environment
env = gym.make(envname)
obssize = env.observation_space.low.size
actsize = env.action_space.n

# initialize policy network
actor = Policy(obssize, actsize, alpha)

# tracking
lrecord = []
episode_losses = []
rrecord = []
total_env_steps = 0

# main iteration
for episode in range(episodes):

    states = []
    actions = []
    preds = []
    rewards = []
    Qhat = []

    state, info = env.reset()
    done = False
    rsum = 0
    t = 0

    while not done:
        total_env_steps += 1
        t += 1

        ########### EPISODE START ###############

        # pick a random action according to the current policy
        prob = actor.compute_prob(state).detach().cpu().numpy().flatten()
        prob = prob / np.sum(prob)
        action = np.random.choice(actsize, p=prob)

        # take action, observe reward and next state
        newstate, r, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # store data
        states.append(state)
        actions.append(action)
        rewards.append(r)

        # prediction: log pi(state, action)
        log_prob_selected = actor.compute_log_prob_selected(state, action)
        preds.append(log_prob_selected)

        rsum += r
        state = newstate

        ############# EPISODE END ################

    # compute discounted returns for the full episode
    Qhat = discounted_rewards(rewards, gamma)

    # subtract baseline and normalize for stability
    Qhat = np.array(discounted_rewards(rewards, gamma), dtype=np.float32)
    Qhat = np.array(discounted_rewards(rewards, gamma), dtype=np.float32)
    Qhat = Qhat - np.mean(Qhat)
    Qhat = Qhat / (np.std(Qhat) + 1e-8)

    preds_tensor = torch.stack(preds)         # shape [T]
    Qhat_tensor = torch.FloatTensor(Qhat)     # shape [T]

    loss = -(preds_tensor * Qhat_tensor).mean()

    actor.optimizer.zero_grad()
    loss.backward()
    actor.optimizer.step()

    episode_losses.append(loss.detach().cpu().item())

    # logging
    rrecord.append(rsum)
    if len(episode_losses) > 0:
        lrecord.append(np.mean(episode_losses))

    if episode % 10 == 0 and episode > 0:
        avg_reward_last_10 = np.mean(rrecord[-10:])
        avg_loss_last_10 = np.mean(lrecord[-10:]) if len(lrecord) > 0 else 0
        print(f"Episode {episode}: Average Reward (last 10) = {avg_reward_last_10:.2f}, Average Loss (last 10) = {avg_loss_last_10:.8e}")

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


Episode 10: Average Reward (last 10) = 25.50, Average Loss (last 10) = -5.42947708e-03
Episode 20: Average Reward (last 10) = 18.60, Average Loss (last 10) = 1.61183435e-02
Episode 30: Average Reward (last 10) = 20.10, Average Loss (last 10) = 1.18585998e-02
Episode 40: Average Reward (last 10) = 24.50, Average Loss (last 10) = 8.37005252e-03
Episode 50: Average Reward (last 10) = 33.30, Average Loss (last 10) = 5.32967945e-03
Episode 60: Average Reward (last 10) = 32.20, Average Loss (last 10) = 3.19733915e-03
Episode 70: Average Reward (last 10) = 28.00, Average Loss (last 10) = 1.87517853e-03
Episode 80: Average Reward (last 10) = 28.00, Average Loss (last 10) = 1.61936544e-03
Episode 90: Average Reward (last 10) = 34.80, Average Loss (last 10) = -1.34448008e-03
Episode 100: Average Reward (last 10) = 43.40, Average Loss (last 10) = -2.28017538e-03
Episode 110: Average Reward (last 10) = 25.90, Average Loss (last 10) = -2.48889095e-03
Episode 120: Average Reward (last 10) = 27.20, A

## Evaluate

The code below is to evaluate the trained model over different random seeds.

In [5]:

def evaluate(policy, env, episodes, seed=None):

    # Set random seeds for reproducibility if seed is provided
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)


    # main iteration
    score = 0
    for episode in range(episodes):

        state, info = env.reset()
        done = False
        rsum = 0

        while not done:

            #pick a random action according to the current policy
            prob = policy.compute_prob(state).cpu().data.numpy()
            prob /= np.sum(prob)
            #action = np.random.choice(env.action_space.n, p=prob.flatten(), size=1).squeeze(0)
            action = np.argmax(prob.flatten())


            # env stepping forward
            newstate, r, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # update data
            rsum += r
            state = newstate



        score +=rsum
    score = score/episodes

    return score

In [6]:
# after training, we will evaluate the performance of the learned policy "actor"
# on a target environment

num_evaluations = 5
eval_episodes = 1000
random_seeds = [42 + i for i in range(num_evaluations)] # Generate 10 different seeds
eval_scores = []

env_test_seeded = gym.make(envname)

print(f"Evaluating agent over {num_evaluations} runs with different seeds...")
for i, seed in enumerate(random_seeds):
    print(f"  Running evaluation {i+1}/{num_evaluations} with seed: {seed}")
    score = evaluate(actor, env_test_seeded, eval_episodes, seed=seed)
    eval_scores.append(score)
    print(f"    Score for seed {seed}: {score:.2f}")

mean_score = np.mean(eval_scores)
std_score = np.std(eval_scores)

print("\n--- Evaluation Results ---")
print(f"Mean evaluation score over {num_evaluations} runs: {mean_score:.2f}")
print(f"Standard deviation of evaluation scores: {std_score:.2f}")

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


Evaluating agent over 5 runs with different seeds...
  Running evaluation 1/5 with seed: 42
    Score for seed 42: 200.00
  Running evaluation 2/5 with seed: 43
    Score for seed 43: 200.00
  Running evaluation 3/5 with seed: 44
    Score for seed 44: 200.00
  Running evaluation 4/5 with seed: 45
    Score for seed 45: 200.00
  Running evaluation 5/5 with seed: 46
    Score for seed 46: 200.00

--- Evaluation Results ---
Mean evaluation score over 5 runs: 200.00
Standard deviation of evaluation scores: 0.00


## Conclusion:

For CartPole-v0, a score of 200 is the maximum episode length / reward cap, which suggests that the policy learned a strong action rule, and the
evaluation is stable across seeds.